# Day 2 - Session 3: tidyverse Universe
**Duration: ~2-2.5 hours**

## Learning Objectives
- Understand the tidyverse philosophy and ecosystem
- Master dplyr for data manipulation
- Learn tidyr for data reshaping
- Perform grouped operations and summaries
- Conduct exploratory data analysis with tidyverse tools
- Create analysis pipelines

## Part 1: Introduction to the tidyverse

### What is the tidyverse?

The **tidyverse** is a collection of R packages designed for data science. All packages share an underlying design philosophy, grammar, and data structures.

**Core tidyverse packages:**
- **dplyr**: Data manipulation
- **tidyr**: Data tidying
- **ggplot2**: Data visualization (covered yesterday)
- **readr**: Reading data
- **purrr**: Functional programming
- **tibble**: Modern data frames
- **stringr**: String manipulation
- **forcats**: Factor handling

### The tidyverse Philosophy

1. **Reuse existing data structures** (data frames/tibbles)
2. **Compose simple functions with pipes**
3. **Embrace functional programming**
4. **Design for humans**

In [ ]:
# Install and load tidyverse
# install.packages("tidyverse")  # Run once

library(tidyverse)  # Loads all core tidyverse packages

# Load sample data
data(starwars)
data(iris)
data(mtcars)

---

## Part 2: Data Manipulation with dplyr

### 2.1 The Pipe Operator: %>%

The pipe operator is key to readable code. It passes output from one function to the next.

**Think of it as "then":** Take data **then** filter **then** select **then** summarize

**Keyboard shortcut:** Ctrl+Shift+M (Windows/Linux) or Cmd+Shift+M (Mac)

In [ ]:
# Without pipe (nested, hard to read)
mean(sqrt(abs(c(-4, -1, 0, 1, 4))))

# With pipe (reads left to right)
c(-4, -1, 0, 1, 4) %>%
  abs() %>%
  sqrt() %>%
  mean()

### 2.2 The Five Main dplyr Verbs

#### filter() - Keep Rows That Match Conditions

In [ ]:
# Filter humans
starwars %>%
  filter(species == "Human") %>%
  head()

# Multiple conditions
starwars %>%
  filter(species == "Human", height > 180) %>%
  select(name, height, species)

# OR condition
starwars %>%
  filter(species %in% c("Human", "Droid", "Wookiee")) %>%
  count(species)

#### select() - Choose Columns

In [ ]:
# Select specific columns
starwars %>%
  select(name, height, mass) %>%
  head()

# Exclude columns
starwars %>%
  select(-films, -vehicles, -starships) %>%
  head()

# Helper functions
starwars %>%
  select(starts_with("s")) %>%
  head()

#### mutate() - Create or Modify Columns

In [ ]:
# Add BMI column
starwars %>%
  mutate(
    bmi = mass / ((height / 100) ^ 2),
    bmi_category = case_when(
      bmi < 18.5 ~ "Underweight",
      bmi < 25 ~ "Normal",
      bmi < 30 ~ "Overweight",
      TRUE ~ "Obese"
    )
  ) %>%
  select(name, bmi, bmi_category) %>%
  filter(!is.na(bmi)) %>%
  head()

#### arrange() - Sort Rows

In [ ]:
# Sort ascending
starwars %>%
  arrange(height) %>%
  select(name, height) %>%
  head()

# Sort descending
starwars %>%
  arrange(desc(height)) %>%
  select(name, height) %>%
  head()

# Multiple columns
starwars %>%
  arrange(species, desc(height)) %>%
  select(name, species, height) %>%
  head(10)

#### summarize() - Calculate Summary Statistics

In [ ]:
# Single summary
starwars %>%
  summarize(
    mean_height = mean(height, na.rm = TRUE),
    sd_height = sd(height, na.rm = TRUE)
  )

# Multiple summaries
starwars %>%
  summarize(
    n = n(),
    mean_height = mean(height, na.rm = TRUE),
    median_height = median(height, na.rm = TRUE),
    max_height = max(height, na.rm = TRUE)
  )

### 2.3 Grouped Operations: The Power of group_by()

**This is where dplyr truly shines!**

In [ ]:
# Calculate mean height by species
starwars %>%
  group_by(species) %>%
  summarize(
    count = n(),
    mean_height = mean(height, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(mean_height)) %>%
  head(10)

In [ ]:
# Multiple grouping variables
starwars %>%
  group_by(species, sex) %>%
  summarize(
    count = n(),
    mean_height = mean(height, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  filter(count > 1) %>%
  arrange(desc(mean_height))

In [ ]:
# Using mutate with groups (compare to group average)
starwars %>%
  group_by(species) %>%
  mutate(
    species_mean_height = mean(height, na.rm = TRUE),
    height_diff_from_species_avg = height - species_mean_height
  ) %>%
  ungroup() %>%
  select(name, species, height, height_diff_from_species_avg) %>%
  filter(!is.na(height_diff_from_species_avg)) %>%
  arrange(desc(height_diff_from_species_avg)) %>%
  head(10)

### 2.4 count() - Quick Frequency Tables

In [ ]:
# Count by species
starwars %>%
  count(species, sort = TRUE) %>%
  head(10)

# Count by multiple variables
starwars %>%
  count(species, homeworld, sort = TRUE) %>%
  head(10)

# Add proportions
starwars %>%
  count(species) %>%
  mutate(proportion = n / sum(n)) %>%
  arrange(desc(proportion)) %>%
  head(10)

---

## Part 3: Data Tidying with tidyr

### 3.1 Tidy Data Principles

**Tidy data** has three rules:
1. Each variable is a column
2. Each observation is a row
3. Each value is a cell

### 3.2 pivot_longer() - Wide to Long Format

In [ ]:
# Example: Convert iris measurements to long format
iris_long <- iris %>%
  mutate(id = row_number()) %>%
  pivot_longer(
    cols = c(Sepal.Length, Sepal.Width, Petal.Length, Petal.Width),
    names_to = "measurement_type",
    values_to = "value"
  )

head(iris_long, 10)

In [ ]:
# Separate measurement type into part and dimension
iris_long %>%
  separate(measurement_type, into = c("part", "dimension"), sep = "\\.") %>%
  head(10)

### 3.3 pivot_wider() - Long to Wide Format

In [ ]:
# Example: Reshape back to wide
iris_wide <- iris_long %>%
  pivot_wider(
    names_from = measurement_type,
    values_from = value
  )

head(iris_wide)

---

## Part 4: Complete Data Analysis Pipeline

Let's combine everything we've learned into a complete analysis.

In [ ]:
# Analysis: Fuel efficiency by car characteristics
mtcars_analysis <- mtcars %>%
  # Add rownames as a column
  rownames_to_column(var = "car_model") %>%
  # Create categorical variables
  mutate(
    cyl_category = case_when(
      cyl == 4 ~ "4 cylinder",
      cyl == 6 ~ "6 cylinder",
      cyl == 8 ~ "8 cylinder"
    ),
    transmission = ifelse(am == 0, "automatic", "manual"),
    efficiency = case_when(
      mpg >= 25 ~ "High",
      mpg >= 20 ~ "Medium",
      TRUE ~ "Low"
    )
  ) %>%
  # Select relevant columns
  select(car_model, mpg, cyl_category, transmission, efficiency, wt, hp) %>%
  # Arrange by mpg
  arrange(desc(mpg))

head(mtcars_analysis, 10)

In [ ]:
# Summarize by cylinder and transmission
summary_stats <- mtcars_analysis %>%
  group_by(cyl_category, transmission) %>%
  summarize(
    n_cars = n(),
    mean_mpg = round(mean(mpg), 2),
    sd_mpg = round(sd(mpg), 2),
    mean_weight = round(mean(wt), 2),
    mean_hp = round(mean(hp), 1),
    .groups = "drop"
  ) %>%
  arrange(desc(mean_mpg))

print(summary_stats)

### 4.1 Combining with ggplot2

In [ ]:
# Visualize the analysis
mtcars_analysis %>%
  ggplot(aes(x = cyl_category, y = mpg, fill = transmission)) +
  geom_boxplot() +
  labs(
    title = "Fuel Efficiency by Cylinder Count and Transmission",
    x = "Number of Cylinders",
    y = "Miles per Gallon",
    fill = "Transmission"
  ) +
  theme_minimal()

In [ ]:
# Faceted analysis
mtcars_analysis %>%
  ggplot(aes(x = wt, y = mpg, color = transmission)) +
  geom_point(size = 3, alpha = 0.7) +
  geom_smooth(method = "lm", se = FALSE) +
  facet_wrap(~ cyl_category) +
  labs(
    title = "Weight vs Fuel Efficiency",
    x = "Weight (1000 lbs)",
    y = "Miles per Gallon",
    color = "Transmission"
  ) +
  theme_minimal()

---

## Part 5: Advanced dplyr Techniques

### 5.1 across() - Apply Functions to Multiple Columns

In [ ]:
# Summarize multiple columns at once
iris %>%
  group_by(Species) %>%
  summarize(
    across(
      c(Sepal.Length, Sepal.Width, Petal.Length, Petal.Width),
      list(mean = mean, sd = sd)
    ),
    .groups = "drop"
  )

### 5.2 case_when() - Multiple Conditions

In [ ]:
# Create categories based on multiple conditions
iris %>%
  mutate(
    size_category = case_when(
      Petal.Length < 2 ~ "Small",
      Petal.Length < 5 ~ "Medium",
      TRUE ~ "Large"  # Everything else
    ),
    width_category = case_when(
      Petal.Width < 0.5 ~ "Narrow",
      Petal.Width < 1.5 ~ "Medium",
      TRUE ~ "Wide"
    )
  ) %>%
  count(Species, size_category, width_category)

### 5.3 joins - Combining Datasets

In [ ]:
# Create example datasets
patients <- tibble(
  patient_id = 1:5,
  name = c("Alice", "Bob", "Charlie", "Diana", "Eve"),
  age = c(45, 32, 56, 28, 41)
)

measurements <- tibble(
  patient_id = c(1, 2, 3, 3, 4, 6),
  measurement = c(7.2, 8.1, 6.5, 7.8, 5.4, 9.0),
  date = c("2024-01-01", "2024-01-02", "2024-01-03", "2024-01-15", "2024-01-04", "2024-01-05")
)

# Inner join (only matching)
inner_join(patients, measurements, by = "patient_id")

In [ ]:
# Left join (keep all patients)
left_join(patients, measurements, by = "patient_id")

# Right join (keep all measurements)
right_join(patients, measurements, by = "patient_id")

---

## Part 6: Practical Exploratory Data Analysis

Let's perform a complete exploratory analysis using tidyverse tools.

In [ ]:
# Load a dataset
data(mpg)

# 1. Initial exploration
glimpse(mpg)

In [ ]:
# 2. Summary statistics
mpg %>%
  summarize(
    n_cars = n(),
    n_manufacturers = n_distinct(manufacturer),
    n_models = n_distinct(model),
    mean_hwy = mean(hwy),
    mean_cty = mean(cty)
  )

In [ ]:
# 3. Group analysis
mpg_by_class <- mpg %>%
  group_by(class) %>%
  summarize(
    count = n(),
    mean_hwy = round(mean(hwy), 1),
    mean_cty = round(mean(cty), 1),
    mean_displ = round(mean(displ), 1),
    .groups = "drop"
  ) %>%
  arrange(desc(mean_hwy))

print(mpg_by_class)

In [ ]:
# 4. Visualize distributions
mpg %>%
  select(hwy, cty, class) %>%
  pivot_longer(
    cols = c(hwy, cty),
    names_to = "fuel_type",
    values_to = "mpg"
  ) %>%
  ggplot(aes(x = class, y = mpg, fill = fuel_type)) +
  geom_boxplot() +
  labs(
    title = "Fuel Efficiency by Vehicle Class",
    x = "Vehicle Class",
    y = "Miles per Gallon",
    fill = "Fuel Type"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
# 5. Correlation analysis
mpg %>%
  select(where(is.numeric)) %>%
  cor() %>%
  round(2)

---

## Practice Exercises

### Exercise 1: Data Transformation
Using the `starwars` dataset:
1. Filter for characters with known height and mass
2. Calculate BMI for each character
3. Create a category: "Light" (BMI < 25) or "Heavy" (BMI >= 25)
4. Calculate the proportion in each category by species
5. Keep only species with more than 2 characters

In [ ]:
# Your code here

### Exercise 2: Grouped Analysis
Using the `iris` dataset:
1. Calculate mean and SD for all measurements by Species
2. Find which measurement has the highest coefficient of variation (SD/mean) for each species
3. Create a visualization comparing the distributions

In [ ]:
# Your code here

### Exercise 3: Complete Pipeline
Using the `mtcars` dataset:
1. Create categories for weight (light/medium/heavy based on tertiles)
2. Calculate mean mpg for each weight category and cylinder count
3. Create a visualization showing this relationship
4. Add a trend line
5. Save the plot with appropriate dimensions

In [ ]:
# Your code here

## Summary

In this session, you learned:

**tidyverse Philosophy:**
- ✅ Understanding the tidyverse ecosystem
- ✅ Composing operations with pipes

**dplyr Core Verbs:**
- ✅ filter() - Selecting rows
- ✅ select() - Choosing columns
- ✅ mutate() - Creating/modifying columns
- ✅ arrange() - Sorting data
- ✅ summarize() - Calculating statistics

**Advanced Techniques:**
- ✅ group_by() for grouped operations
- ✅ across() for multiple columns
- ✅ case_when() for complex conditions
- ✅ Joining datasets

**tidyr:**
- ✅ pivot_longer() and pivot_wider()
- ✅ Tidy data principles

**Integration:**
- ✅ Combining dplyr with ggplot2
- ✅ Complete analysis pipelines
- ✅ Exploratory data analysis workflows

**Key Takeaway**: The tidyverse provides a consistent, powerful toolkit for modern data analysis in R. The pipe operator and verb-based functions make your code more readable and maintainable.

**Congratulations!** You've completed the Introduction to R course!

## Additional Resources

### Cheat Sheets:
- dplyr: https://github.com/rstudio/cheatsheets/raw/master/data-transformation.pdf
- tidyr: https://github.com/rstudio/cheatsheets/raw/master/tidyr.pdf
- ggplot2: https://github.com/rstudio/cheatsheets/raw/master/data-visualization.pdf

### Books:
- R for Data Science: https://r4ds.had.co.nz/
- tidyverse documentation: https://www.tidyverse.org/

### Community:
- RStudio Community: https://community.rstudio.com/
- R4DS Online Learning Community: https://www.rfordatasci.com/

In [ ]:
sessionInfo()

## Session Info

It's good practice to record your R session information at the end of your analysis. This helps with reproducibility by documenting:
- R version
- Operating system
- Loaded packages and their versions
- Other system details